# 01 — Data Pipeline

Vanguard Quantum for Finance — Multi-Asset Portfolio Construction.

Run this in Colab after cloning the repo. Prices are hybrid-sourced: real OHLCV via a HF/DuckDB dataset for 9 of 15 tickers, yfinance for the rest, and a labeled synthetic generator only if both are unreachable. Builds returns/vol/cost/yield inputs for the rest of the project.

In [ ]:
# --- Colab setup: clone repo + install package ---
# Replace with your actual repo URL once pushed to GitHub.
!git clone https://github.com/YOUR_USERNAME/vanguard-quantum-portfolio.git
%cd vanguard-quantum-portfolio
!pip install -e . -q

In [ ]:
from vqportfolio.market_data.loader import load_prices, load_ohlc_with_sources
from vqportfolio.market_data.overlays import compute_returns_and_risk, compute_cost_and_yield
from vqportfolio.config import TICKERS

prices, used_synthetic = load_prices()
print(f'USED_SYNTHETIC_PRICES = {used_synthetic}')
assert not used_synthetic, 'Running on synthetic fallback — check internet access before trusting results.'

# per-ticker source breakdown: which came from HF/DuckDB vs yfinance vs synthetic
_, sources = load_ohlc_with_sources()
for t, s in sources.items():
    print(f'  {t}: {s}')
prices.tail()

In [ ]:
mu, sigma, log_returns = compute_returns_and_risk(prices)
overlay = compute_cost_and_yield(TICKERS, log_returns)

print('USED_SYNTHETIC_COST  =', overlay.attrs['cost_synthetic'])
print('USED_SYNTHETIC_YIELD =', overlay.attrs['yield_synthetic'])
# cost_bps: real Corwin-Schultz spread estimator on real OHLC when online.
# yield: real trailing dividend yield via yfinance when online.
# Both fall back to a documented synthetic estimate only if unreachable.
mu.sort_values(ascending=False)

## Save processed data

Cache to `data/processed/` so downstream notebooks don't need to re-pull.

In [ ]:
import pandas as pd

prices.to_csv('data/processed/prices.csv')
mu.to_csv('data/processed/mu.csv')
sigma.to_csv('data/processed/sigma.csv')
overlay.to_csv('data/processed/overlay.csv')
print('Saved to data/processed/')